Loading Phi-1.5 and CLIP models

In [ ]:
# defining phi model paths (base & fine-tuned weights)
ADAPTER_PATH = os.path.join(os.path.dirname(__file__), 'content/my_phi_model')
BASE_MODEL = "/mnt/models/phi-1_5_snapshot"
DEVICE = 'cpu'

# initializing phi & clip models, text queries, and image classes
base = None
phi = None
tokenizer = None
clip = None
processor = None

In [ ]:
# load phi base model
try:
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        local_files_only=True,
        low_cpu_mem_usage=True
    )
    base.to(DEVICE)
    base.eval()
except Exception:
    base = None

# load tokenizer
try:
    if os.path.isdir(ADAPTER_PATH) and any(fname.startswith("tokenizer") for fname in os.listdir(ADAPTER_PATH)):
        tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True, use_fast=True, local_files_only=True)
    elif base is not None:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True, local_files_only=True)
    else:
        tokenizer = None
except Exception:
    tokenizer = None

# load clip model & processor
try:
    clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip.to(DEVICE)
    clip.eval()
except Exception:
    clip = None
    processor = None

In [ ]:
# attach adapter (fine-tuned weights) to phi base model
if base is not None and os.path.isdir(ADAPTER_PATH):
    try:
        phi = PeftModel.from_pretrained(base, ADAPTER_PATH, is_trainable=False)
        phi.to(DEVICE)
        phi.eval()
    except Exception:
        phi = base
else:
    phi = base

This code snippet is pulled from my [main.py](../src/api/main.py) file and is used to load the Phi-1.5 and CLIP models into my web app back-end. My fine-tuned model weights are then attached to the Phi-1.5 model.